In [1]:
"""
Unified Pruning Benchmark — ResNet-50 / CIFAR-10
=================================================
Runs all 7 pruning methods, picks the best configuration from each
(highest accuracy within MAX_ACC_DROP=0.02 threshold), and saves
a single consolidated JSON report.

Output: pruning_benchmark_results.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "pruning_benchmark_results.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY_LEVELS      = [0.30, 0.50, 0.70, 0.80, 0.90]
STRUCTURED_RATIOS    = [0.10, 0.20, 0.30, 0.40, 0.50]
MAX_ACC_DROP         = 0.02
CALIBRATION_BATCHES  = 10
ITERATIVE_ROUNDS_LTH = 5
PRUNING_STEP_ITER    = 0.15
MAX_SPARSITY_ITER    = 0.90
INFERENCE_RUNS       = 50


# ══════════════════════════════════════════════════════════════════════════════
# SHARED UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    active = sum((p != 0).sum().item() for p in model.parameters())
    return total, active

def sparse_size_mb(model):
    active = sum((p != 0).sum().item() for p in model.parameters())
    return active * 4 / 1024**2

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)
    return size

def compute_flops(model):
    """
    Estimate FLOPs for ResNet-50 on 32x32 input using layer-by-layer counting.
    FLOPs for Conv2d  = 2 * C_in * C_out * kH * kW * oH * oW
    FLOPs for Linear  = 2 * in_features * out_features
    Active FLOPs scale by (1 - sparsity) for pruned models.
    """
    total_flops = 0
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    hooks = []
    flop_counts = {}

    def make_hook(name):
        def hook(module, inp, out):
            if isinstance(module, nn.Conv2d):
                b, c_out, oH, oW = out.shape
                c_in  = module.in_channels
                kH, kW = module.kernel_size
                # multiply-add = 2 ops per multiply
                flops = 2 * c_in * c_out * kH * kW * oH * oW
                flop_counts[name] = flops
            elif isinstance(module, nn.Linear):
                flops = 2 * module.in_features * module.out_features
                flop_counts[name] = flops
        return hook

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            hooks.append(module.register_forward_hook(make_hook(name)))

    model.eval()
    with torch.no_grad():
        model(dummy)

    for h in hooks:
        h.remove()

    total_flops = sum(flop_counts.values())
    return total_flops

def compute_active_flops(model):
    """
    FLOPs scaled by weight density — approximates active multiply-adds
    after pruning (zero weights skip computation in ideal sparse hardware).
    """
    total_flops = 0
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    hooks = []
    flop_counts   = {}
    density_cache = {}

    # Pre-compute per-layer density
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.data
            density_cache[name] = (w != 0).float().mean().item()

    def make_hook(name):
        def hook(module, inp, out):
            if isinstance(module, nn.Conv2d):
                b, c_out, oH, oW = out.shape
                c_in  = module.in_channels
                kH, kW = module.kernel_size
                flops = 2 * c_in * c_out * kH * kW * oH * oW
                flop_counts[name] = flops * density_cache.get(name, 1.0)
            elif isinstance(module, nn.Linear):
                flops = 2 * module.in_features * module.out_features
                flop_counts[name] = flops * density_cache.get(name, 1.0)
        return hook

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            hooks.append(module.register_forward_hook(make_hook(name)))

    model.eval()
    with torch.no_grad():
        model(dummy)

    for h in hooks:
        h.remove()

    return sum(flop_counts.values())

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_inference_ms(model):
    """CPU inference time (ms) per batch."""
    cpu_m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5): cpu_m(dummy)
        t0 = time.perf_counter()
        for _ in range(INFERENCE_RUNS): cpu_m(dummy)
    return (time.perf_counter() - t0) / INFERENCE_RUNS * 1000

def build_best_entry(pruned_model, metrics, baseline_acc, config_label, config_detail):
    """Build the standardised result dict for the final JSON."""
    total_p, active_p = count_params(pruned_model)
    sp_mb   = sparse_size_mb(pruned_model)
    inf_ms  = measure_inference_ms(pruned_model)
    active_flops = compute_active_flops(pruned_model)
    acc_drop = baseline_acc - metrics["accuracy"]

    return {
        "best_config"       : config_label,
        "config_detail"     : config_detail,
        "accuracy"          : round(metrics["accuracy"], 6),
        "precision"         : round(metrics["precision"], 6),
        "recall"            : round(metrics["recall"], 6),
        "f1_score"          : round(metrics["f1"], 6),
        "accuracy_drop"     : round(acc_drop, 6),
        "params_total"      : total_p,
        "params_active"     : active_p,
        "model_size_mb"     : round(sp_mb, 4),
        "inference_time_ms" : round(inf_ms, 4),
        "flops_active"      : int(active_flops),
        "sparsity"          : round(real_sparsity(pruned_model), 4),
    }

def score_candidate(metrics, base_disk, dk_mb, acc_drop):
    if acc_drop > MAX_ACC_DROP:
        return -1.0
    return metrics["accuracy"] + ((base_disk - dk_mb) / base_disk) * 0.1


# ══════════════════════════════════════════════════════════════════════════════
# METHOD 1 — UNSTRUCTURED PRUNING (Global L1)
# ══════════════════════════════════════════════════════════════════════════════

def run_unstructured(model, loader, baseline_acc, base_disk):
    print("\n" + "="*60)
    print("  [1/7] Unstructured Pruning (Global L1)")
    print("="*60)

    best_model, best_score, best_sp = None, -1.0, None

    for sparsity in SPARSITY_LEVELS:
        pruned = copy.deepcopy(model)
        layers = prunable_layers(pruned)
        prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=sparsity)
        for m, p in layers: prune.remove(m, p)

        metrics  = evaluate(pruned, loader)
        acc_drop = baseline_acc - metrics["accuracy"]
        dk_mb    = disk_size_mb(pruned)
        s        = score_candidate(metrics, base_disk, dk_mb, acc_drop)

        print(f"  sp={int(sparsity*100)}% | acc={metrics['accuracy']:.4f} | drop={acc_drop:+.4f}")

        if s > best_score:
            best_score, best_model, best_sp = s, copy.deepcopy(pruned), sparsity

    if best_model is None:
        print("  WARNING: no config within acc-drop threshold — using least-bad")
        # fall back: pick highest accuracy regardless
        best_model = copy.deepcopy(model)
        best_sp    = 0.0

    metrics = evaluate(best_model, loader)
    return build_best_entry(
        best_model, metrics, baseline_acc,
        f"sparsity={int(best_sp*100)}%",
        {"target_sparsity": best_sp, "criterion": "global_L1"}
    )


# ══════════════════════════════════════════════════════════════════════════════
# METHOD 2 — STRUCTURED PRUNING (L1 Filter)
# ══════════════════════════════════════════════════════════════════════════════

def run_structured(model, loader, baseline_acc, base_disk):
    print("\n" + "="*60)
    print("  [2/7] Structured Pruning (L1 Filter)")
    print("="*60)

    best_model, best_score, best_ratio = None, -1.0, None

    for ratio in STRUCTURED_RATIOS:
        pruned = copy.deepcopy(model)
        for name, module in pruned.named_modules():
            if isinstance(module, nn.Conv2d) and module.weight.shape[0] > 1:
                n = max(1, min(int(module.weight.shape[0] * ratio),
                               module.weight.shape[0] - 1))
                prune.ln_structured(module, name="weight", amount=n, n=1, dim=0)
                prune.remove(module, "weight")

        metrics  = evaluate(pruned, loader)
        acc_drop = baseline_acc - metrics["accuracy"]
        dk_mb    = disk_size_mb(pruned)
        s        = score_candidate(metrics, base_disk, dk_mb, acc_drop)

        print(f"  ratio={int(ratio*100)}% | acc={metrics['accuracy']:.4f} | drop={acc_drop:+.4f}")

        if s > best_score:
            best_score, best_model, best_ratio = s, copy.deepcopy(pruned), ratio

    if best_model is None:
        best_model, best_ratio = copy.deepcopy(model), 0.0

    metrics = evaluate(best_model, loader)
    return build_best_entry(
        best_model, metrics, baseline_acc,
        f"filter_ratio={int(best_ratio*100)}%",
        {"filter_pruning_ratio": best_ratio, "criterion": "L1_filter_norm"}
    )


# ══════════════════════════════════════════════════════════════════════════════
# METHOD 3 — MAGNITUDE PRUNING (Local L1 / L2 / Global L1)
# ══════════════════════════════════════════════════════════════════════════════

def _mag_l1_local(model, sparsity):
    pruned = copy.deepcopy(model)
    for m, p in prunable_layers(pruned):
        prune.l1_unstructured(m, name=p, amount=sparsity)
        prune.remove(m, p)
    return pruned

def _mag_l2_local(model, sparsity):
    pruned = copy.deepcopy(model)
    for _, module in pruned.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w   = module.weight.data
            flat = w.abs().pow(2).view(-1)
            thr = torch.kthvalue(flat, max(1, int(flat.numel() * sparsity))).values
            module.weight.data = w * (w.abs().pow(2) > thr).float()
    return pruned

def _mag_l1_global(model, sparsity):
    pruned = copy.deepcopy(model)
    layers = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=sparsity)
    for m, p in layers: prune.remove(m, p)
    return pruned

def run_magnitude(model, loader, baseline_acc, base_disk):
    print("\n" + "="*60)
    print("  [3/7] Magnitude Pruning (local-L1, local-L2, global-L1)")
    print("="*60)

    criteria = [
        ("local_l1",  _mag_l1_local),
        ("local_l2",  _mag_l2_local),
        ("global_l1", _mag_l1_global),
    ]
    best_model, best_score, best_cfg = None, -1.0, {}

    for sparsity in SPARSITY_LEVELS:
        for cname, fn in criteria:
            pruned   = fn(model, sparsity)
            metrics  = evaluate(pruned, loader)
            acc_drop = baseline_acc - metrics["accuracy"]
            dk_mb    = disk_size_mb(pruned)
            s        = score_candidate(metrics, base_disk, dk_mb, acc_drop)
            print(f"  sp={int(sparsity*100)}% {cname} | acc={metrics['accuracy']:.4f} | drop={acc_drop:+.4f}")
            if s > best_score:
                best_score, best_model = s, copy.deepcopy(pruned)
                best_cfg = {"sparsity": sparsity, "criterion": cname}

    if best_model is None:
        best_model = copy.deepcopy(model)
        best_cfg   = {"sparsity": 0.0, "criterion": "none"}

    metrics = evaluate(best_model, loader)
    return build_best_entry(
        best_model, metrics, baseline_acc,
        f"{best_cfg['criterion']} @ {int(best_cfg['sparsity']*100)}%",
        best_cfg
    )


# ══════════════════════════════════════════════════════════════════════════════
# METHOD 4 — MOVEMENT PRUNING (Taylor |w*grad|)
# ══════════════════════════════════════════════════════════════════════════════

def _compute_taylor_scores(model, loader, n_batches):
    model.train()
    criterion = nn.CrossEntropyLoss()
    model.zero_grad()
    n = 0
    for i, (inputs, targets) in enumerate(loader):
        if i >= n_batches: break
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        criterion(model(inputs), targets).backward()
        n += 1
    scores = {}
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and module.weight.grad is not None:
            scores[name] = (module.weight.data * module.weight.grad / n).abs().detach().cpu()
    model.zero_grad(); model.eval()
    return scores

def _apply_movement(model, loader, sparsity):
    pruned = copy.deepcopy(model)
    scores = _compute_taylor_scores(pruned, loader, CALIBRATION_BATCHES)
    if not scores:
        layers = prunable_layers(pruned)
        prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=sparsity)
        for m, p in layers: prune.remove(m, p)
        return pruned
    all_s = torch.cat([s.view(-1) for s in scores.values()])
    thr   = torch.kthvalue(all_s, max(1, int(all_s.numel() * sparsity))).values.item()
    with torch.no_grad():
        for name, module in pruned.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)) and name in scores:
                module.weight.data *= (scores[name].to(DEVICE) > thr).float()
    return pruned

def run_movement(model, loader, baseline_acc, base_disk):
    print("\n" + "="*60)
    print("  [4/7] Movement Pruning (Taylor |w*grad|)")
    print("="*60)

    best_model, best_score, best_sp = None, -1.0, None
    for sparsity in SPARSITY_LEVELS:
        pruned   = _apply_movement(model, loader, sparsity)
        metrics  = evaluate(pruned, loader)
        acc_drop = baseline_acc - metrics["accuracy"]
        dk_mb    = disk_size_mb(pruned)
        s        = score_candidate(metrics, base_disk, dk_mb, acc_drop)
        print(f"  sp={int(sparsity*100)}% | acc={metrics['accuracy']:.4f} | drop={acc_drop:+.4f}")
        if s > best_score:
            best_score, best_model, best_sp = s, copy.deepcopy(pruned), sparsity

    if best_model is None:
        best_model, best_sp = copy.deepcopy(model), 0.0

    metrics = evaluate(best_model, loader)
    return build_best_entry(
        best_model, metrics, baseline_acc,
        f"sparsity={int(best_sp*100)}%",
        {"target_sparsity": best_sp, "calibration_batches": CALIBRATION_BATCHES,
         "scoring": "Taylor |w*grad|"}
    )


# ══════════════════════════════════════════════════════════════════════════════
# METHOD 5 — LOTTERY TICKET HYPOTHESIS (IMP)
# ══════════════════════════════════════════════════════════════════════════════

def _find_ticket(trained_model, target_sparsity, n_rounds):
    per_round = 1.0 - (1.0 - target_sparsity) ** (1.0 / n_rounds)
    current   = copy.deepcopy(trained_model)
    for _ in range(n_rounds):
        layers = prunable_layers(current)
        prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=per_round)
        for m, p in layers: prune.remove(m, p)
    return current

def run_lottery_ticket(model, loader, baseline_acc, base_disk):
    print("\n" + "="*60)
    print("  [5/7] Lottery Ticket Hypothesis")
    print("="*60)

    best_model, best_score, best_sp = None, -1.0, None
    for sparsity in SPARSITY_LEVELS:
        ticket_model = _find_ticket(model, sparsity, ITERATIVE_ROUNDS_LTH)
        mask = {name: (m.weight.data != 0).float()
                for name, m in ticket_model.named_modules()
                if isinstance(m, (nn.Conv2d, nn.Linear))}

        winning = copy.deepcopy(model)
        for name, module in winning.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)) and name in mask:
                module.weight.data *= mask[name].to(DEVICE)

        metrics  = evaluate(winning, loader)
        acc_drop = baseline_acc - metrics["accuracy"]
        dk_mb    = disk_size_mb(winning)
        s        = score_candidate(metrics, base_disk, dk_mb, acc_drop)
        print(f"  sp={int(sparsity*100)}% | acc={metrics['accuracy']:.4f} | drop={acc_drop:+.4f}")
        if s > best_score:
            best_score, best_model, best_sp = s, copy.deepcopy(winning), sparsity

    if best_model is None:
        best_model, best_sp = copy.deepcopy(model), 0.0

    metrics = evaluate(best_model, loader)
    return build_best_entry(
        best_model, metrics, baseline_acc,
        f"sparsity={int(best_sp*100)}%",
        {"target_sparsity": best_sp, "iterative_rounds": ITERATIVE_ROUNDS_LTH,
         "ticket_type": "trained_weights_x_mask"}
    )


# ══════════════════════════════════════════════════════════════════════════════
# METHOD 6 — ITERATIVE PRUNING
# ══════════════════════════════════════════════════════════════════════════════

def run_iterative(model, loader, baseline_acc, base_disk):
    print("\n" + "="*60)
    print("  [6/7] Iterative Pruning")
    print("="*60)

    current   = copy.deepcopy(model)
    best_model, best_score, best_sp = None, -1.0, None
    round_num = 0

    while True:
        round_num += 1
        # Prune PRUNING_STEP fraction of current non-zero weights
        all_w = torch.cat([
            m.weight.data[m.weight.data != 0].abs().view(-1)
            for _, m in current.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))
        ])
        if all_w.numel() == 0: break
        k   = max(1, int(all_w.numel() * PRUNING_STEP_ITER))
        thr = torch.kthvalue(all_w, k).values.item()
        with torch.no_grad():
            for _, module in current.named_modules():
                if isinstance(module, (nn.Conv2d, nn.Linear)):
                    module.weight.data *= (module.weight.data.abs() > thr).float()

        actual_sp = real_sparsity(current)
        metrics   = evaluate(current, loader)
        acc_drop  = baseline_acc - metrics["accuracy"]
        dk_mb     = disk_size_mb(current)
        s         = score_candidate(metrics, base_disk, dk_mb, acc_drop)
        print(f"  round={round_num} sp={actual_sp*100:.1f}% | acc={metrics['accuracy']:.4f} | drop={acc_drop:+.4f}")

        if s > best_score:
            best_score, best_model, best_sp = s, copy.deepcopy(current), actual_sp

        if actual_sp >= MAX_SPARSITY_ITER or round_num > 30: break

    if best_model is None:
        best_model, best_sp = copy.deepcopy(model), 0.0

    metrics = evaluate(best_model, loader)
    return build_best_entry(
        best_model, metrics, baseline_acc,
        f"sparsity={best_sp*100:.1f}%",
        {"cumulative_sparsity": best_sp, "step_per_round": PRUNING_STEP_ITER,
         "total_rounds_run": round_num}
    )


# ══════════════════════════════════════════════════════════════════════════════
# METHOD 7 — ONE-SHOT PRUNING (L1, L2, Random)
# ══════════════════════════════════════════════════════════════════════════════

class _L2Unstructured(prune.BasePruningMethod):
    PRUNING_TYPE = "unstructured"
    def compute_mask(self, t, default_mask):
        n = t.nelement()
        n_prune = max(1, round(n * self.amount))
        mask = default_mask.clone(memory_format=torch.contiguous_format)
        topk = torch.topk(t.pow(2).view(-1), k=n - n_prune, largest=True)
        mask.view(-1).fill_(0)
        mask.view(-1)[topk.indices] = 1
        return mask
    @classmethod
    def apply(cls, module, name, amount):
        return super().apply(module, name, amount=amount)

def run_oneshot(model, loader, baseline_acc, base_disk):
    print("\n" + "="*60)
    print("  [7/7] One-Shot Pruning (L1-global, L2-global, Random)")
    print("="*60)

    variants = {
        "l1_global": lambda m, s: _oneshot_l1(m, s),
        "l2_global": lambda m, s: _oneshot_l2(m, s),
        "random":    lambda m, s: _oneshot_rand(m, s),
    }

    def _oneshot_l1(m, s):
        p = copy.deepcopy(m)
        layers = prunable_layers(p)
        prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=s)
        for mod, param in layers: prune.remove(mod, param)
        return p

    def _oneshot_l2(m, s):
        p = copy.deepcopy(m)
        for mod, param in prunable_layers(p):
            _L2Unstructured.apply(mod, param, amount=s)
            prune.remove(mod, param)
        return p

    def _oneshot_rand(m, s):
        p = copy.deepcopy(m)
        layers = prunable_layers(p)
        prune.global_unstructured(layers, pruning_method=prune.RandomUnstructured, amount=s)
        for mod, param in layers: prune.remove(mod, param)
        return p

    variants = [
        ("l1_global",  _oneshot_l1),
        ("l2_global",  _oneshot_l2),
        ("random",     _oneshot_rand),
    ]

    best_model, best_score, best_cfg = None, -1.0, {}
    for sparsity in SPARSITY_LEVELS:
        for vname, fn in variants:
            try:
                pruned = fn(model, sparsity)
            except Exception as e:
                print(f"  ERROR {vname} sp={sparsity}: {e}")
                continue
            metrics  = evaluate(pruned, loader)
            acc_drop = baseline_acc - metrics["accuracy"]
            dk_mb    = disk_size_mb(pruned)
            s        = score_candidate(metrics, base_disk, dk_mb, acc_drop)
            print(f"  sp={int(sparsity*100)}% {vname} | acc={metrics['accuracy']:.4f} | drop={acc_drop:+.4f}")
            if s > best_score:
                best_score, best_model = s, copy.deepcopy(pruned)
                best_cfg = {"sparsity": sparsity, "variant": vname}

    if best_model is None:
        best_model = copy.deepcopy(model)
        best_cfg   = {"sparsity": 0.0, "variant": "none"}

    metrics = evaluate(best_model, loader)
    return build_best_entry(
        best_model, metrics, baseline_acc,
        f"{best_cfg['variant']} @ {int(best_cfg['sparsity']*100)}%",
        best_cfg
    )


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print(f"\n{'#'*64}")
    print("  Unified Pruning Benchmark — ResNet-50 / CIFAR-10")
    print(f"  Device: {DEVICE}")
    print(f"{'#'*64}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()

    base_disk  = disk_size_mb(model)
    base_flops = compute_flops(model)
    total_p, _ = count_params(model)
    base_inf   = measure_inference_ms(model)

    print(f"\n  Baseline: acc={baseline_acc:.4f} | params={total_p:,} | "
          f"size={base_disk:.1f}MB | FLOPs={base_flops:,.0f} | inf={base_inf:.1f}ms\n")

    results = {}

    results["baseline"] = {
        "accuracy"          : round(baseline_acc, 6),
        "precision"         : round(baseline.get("precision", 0), 6),
        "recall"            : round(baseline.get("recall", 0), 6),
        "f1_score"          : round(baseline.get("f1", 0), 6),
        "accuracy_drop"     : 0.0,
        "params_total"      : total_p,
        "params_active"     : total_p,
        "model_size_mb"     : round(base_disk, 4),
        "inference_time_ms" : round(base_inf, 4),
        "flops_active"      : int(base_flops),
        "sparsity"          : 0.0,
    }

    results["unstructured_pruning"]   = run_unstructured(model, loader, baseline_acc, base_disk)
    results["structured_pruning"]     = run_structured(model, loader, baseline_acc, base_disk)
    results["magnitude_pruning"]      = run_magnitude(model, loader, baseline_acc, base_disk)
    results["movement_pruning"]       = run_movement(model, loader, baseline_acc, base_disk)
    results["lottery_ticket"]         = run_lottery_ticket(model, loader, baseline_acc, base_disk)
    results["iterative_pruning"]      = run_iterative(model, loader, baseline_acc, base_disk)
    results["oneshot_pruning"]        = run_oneshot(model, loader, baseline_acc, base_disk)

    # ── Summary table ──────────────────────────────────────────────────────────
    print("\n\n" + "="*80)
    print("  FINAL SUMMARY — Best Result per Method")
    print("="*80)
    hdr = f"{'Method':<28} {'Acc':>7} {'Drop':>7} {'Sparsity':>9} {'Params(M)':>10} {'Size(MB)':>9} {'Inf(ms)':>8} {'FLOPs(M)':>9}"
    print(hdr)
    print("-"*80)
    for method, r in results.items():
        print(f"  {method:<26} {r['accuracy']:>7.4f} {r['accuracy_drop']:>+7.4f} "
              f"{r['sparsity']*100:>8.1f}% {r['params_active']/1e6:>9.2f}M "
              f"{r['model_size_mb']:>8.1f} {r['inference_time_ms']:>7.1f} "
              f"{r['flops_active']/1e6:>8.1f}M")

    with open(OUTPUT_JSON, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n  ✓ Saved -> {OUTPUT_JSON}")


if __name__ == "__main__":
    main()


################################################################
  Unified Pruning Benchmark — ResNet-50 / CIFAR-10
  Device: cuda
################################################################


  Baseline: acc=0.9320 | params=23,520,842 | size=90.0MB | FLOPs=2,595,659,776 | inf=748.9ms


  [1/7] Unstructured Pruning (Global L1)
  sp=30% | acc=0.9321 | drop=-0.0001
  sp=50% | acc=0.9320 | drop=+0.0000
  sp=70% | acc=0.9319 | drop=+0.0001
  sp=80% | acc=0.9276 | drop=+0.0044
  sp=90% | acc=0.8774 | drop=+0.0546

  [2/7] Structured Pruning (L1 Filter)
  ratio=10% | acc=0.9322 | drop=-0.0002
  ratio=20% | acc=0.9310 | drop=+0.0010
  ratio=30% | acc=0.9313 | drop=+0.0007
  ratio=40% | acc=0.9227 | drop=+0.0093
  ratio=50% | acc=0.8734 | drop=+0.0586

  [3/7] Magnitude Pruning (local-L1, local-L2, global-L1)
  sp=30% local_l1 | acc=0.9319 | drop=+0.0001
  sp=30% local_l2 | acc=0.9319 | drop=+0.0001
  sp=30% global_l1 | acc=0.9321 | drop=-0.0001
  sp=50% local_l1 | acc=0.9318 | drop=+0.0